# 04. Neo4j vs AGE — Side-by-Side Comparison

동일한 작업을 `langchain-neo4j`와 `langchain-age`로 나란히 실행하여 API 호환성을 비교.

| | Neo4j | AGE |
|---|---|---|
| DB | Neo4j (Bolt) | PostgreSQL + Apache AGE |
| Vector | Native index | pgvector |
| Package | `langchain-neo4j` | `langchain-age` |

**사전 준비**
```bash
# AGE
cd docker && docker compose up -d

# Neo4j (별도 컨테이너)
docker run -d --name neo4j-test \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:5

pip install "langchain-age[all]" langchain-neo4j langchain-openai
```

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Connection strings
AGE_DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"
NEO4J_URL = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"

---
## 1. Graph — 연결 및 노드 생성

**동일한 Cypher를 두 시스템에서 실행한다.**

In [ ]:
# ── Neo4j ──
from langchain_neo4j import Neo4jGraph

neo4j_graph = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USER,
    password=NEO4J_PASS,
)
print(f"Neo4j: connected")

In [ ]:
# ── AGE ──
from langchain_age import AGEGraph

age_graph = AGEGraph(
    connection_string=AGE_DSN,
    graph_name="comparison",
)
print(f"AGE: connected")

In [ ]:
# ── 동일한 Cypher ──
cypher_statements = [
    "MERGE (:Movie {title: 'The Matrix', year: 1999})",
    "MERGE (:Movie {title: 'Inception', year: 2010})",
    "MERGE (:Person {name: 'Keanu Reeves'})",
    "MERGE (:Person {name: 'Leonardo DiCaprio'})",
    "MATCH (p:Person {name: 'Keanu Reeves'}), (m:Movie {title: 'The Matrix'}) MERGE (p)-[:ACTED_IN]->(m)",
    "MATCH (p:Person {name: 'Leonardo DiCaprio'}), (m:Movie {title: 'Inception'}) MERGE (p)-[:ACTED_IN]->(m)",
]

for stmt in cypher_statements:
    neo4j_graph.query(stmt)
    age_graph.query(stmt)

print("Both: nodes and relationships created.")

## 2. Graph — 쿼리 결과 비교

In [ ]:
query = "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p.name AS actor, m.title AS movie"

print("=== Neo4j ===")
for r in neo4j_graph.query(query):
    print(f"  {r['actor']} → {r['movie']}")

print("\n=== AGE ===")
for r in age_graph.query(query):
    print(f"  {r['actor']} → {r['movie']}")

## 3. Graph — 스키마 비교

In [ ]:
neo4j_graph.refresh_schema()
age_graph.refresh_schema()

print("=== Neo4j Schema ===")
print(neo4j_graph.schema[:500])

print("\n=== AGE Schema ===")
print(age_graph.schema[:500])

---
## 4. Vector — 동일한 문서 벡터화

In [ ]:
texts = [
    "The Matrix is a sci-fi action film about simulated reality.",
    "Inception explores dreams within dreams and the nature of consciousness.",
    "Keanu Reeves is known for action roles in The Matrix and John Wick.",
    "Leonardo DiCaprio starred in Titanic, Inception, and The Revenant.",
    "Science fiction films often explore technology and human identity.",
]
metadatas = [
    {"type": "movie", "genre": "sci-fi"},
    {"type": "movie", "genre": "sci-fi"},
    {"type": "person", "role": "actor"},
    {"type": "person", "role": "actor"},
    {"type": "topic", "genre": "sci-fi"},
]

In [ ]:
# ── Neo4j Vector ──
from langchain_neo4j import Neo4jVector

neo4j_store = Neo4jVector.from_texts(
    texts,
    embedding=embeddings,
    metadatas=metadatas,
    url=NEO4J_URL,
    username=NEO4J_USER,
    password=NEO4J_PASS,
)
print(f"Neo4j: {len(texts)} vectors stored")

In [ ]:
# ── AGE Vector ──
from langchain_age import AGEVector, DistanceStrategy

age_store = AGEVector.from_texts(
    texts,
    embedding=embeddings,
    metadatas=metadatas,
    connection_string=AGE_DSN,
    collection_name="comparison_vectors",
    distance_strategy=DistanceStrategy.COSINE,
    pre_delete_collection=True,
)
print(f"AGE: {len(texts)} vectors stored")

## 5. Vector — 유사도 검색 비교

In [ ]:
query_text = "sci-fi movies about reality"

print(f"Query: '{query_text}'\n")

print("=== Neo4j ===")
for doc in neo4j_store.similarity_search(query_text, k=3):
    print(f"  {doc.page_content[:70]}...")

print("\n=== AGE ===")
for doc in age_store.similarity_search(query_text, k=3):
    print(f"  {doc.page_content[:70]}...")

## 6. Vector — Retriever로 사용

In [ ]:
neo4j_retriever = neo4j_store.as_retriever(search_kwargs={"k": 2})
age_retriever = age_store.as_retriever(search_kwargs={"k": 2})

query_text = "Who acted in action movies?"

print("=== Neo4j Retriever ===")
for doc in neo4j_retriever.invoke(query_text):
    print(f"  {doc.page_content[:70]}...")

print("\n=== AGE Retriever ===")
for doc in age_retriever.invoke(query_text):
    print(f"  {doc.page_content[:70]}...")

---
## 7. QA Chain 비교

동일한 질문을 Neo4j `GraphCypherQAChain`과 AGE `AGEGraphCypherQAChain`에 실행.

In [ ]:
# ── Neo4j Chain ──
from langchain_neo4j import GraphCypherQAChain

neo4j_chain = GraphCypherQAChain.from_llm(
    llm,
    graph=neo4j_graph,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
)

# ── AGE Chain ──
from langchain_age import AGEGraphCypherQAChain

age_chain = AGEGraphCypherQAChain.from_llm(
    llm,
    graph=age_graph,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
)

print("Both chains ready.")

In [ ]:
questions = [
    "Which movies did Keanu Reeves act in?",
    "Who acted in Inception?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("-" * 60)

    neo4j_result = neo4j_chain.invoke({"query": q})
    print(f"Neo4j  | Cypher: {neo4j_result['intermediate_steps'][0]['query']}")
    print(f"       | Answer: {neo4j_result['result']}")

    age_result = age_chain.invoke({"query": q})
    print(f"AGE    | Cypher: {age_result['intermediate_steps'][0]['query']}")
    print(f"       | Answer: {age_result['result']}")

---
## 8. API 호환성 요약

| Operation | Neo4j | AGE | 동일? |
|-----------|-------|-----|-------|
| `graph.query(cypher)` | `Neo4jGraph.query()` | `AGEGraph.query()` | **동일** |
| `graph.schema` | `Neo4jGraph.schema` | `AGEGraph.schema` | **동일** |
| `graph.refresh_schema()` | `Neo4jGraph.refresh_schema()` | `AGEGraph.refresh_schema()` | **동일** |
| `graph.add_graph_documents()` | `Neo4jGraph.add_graph_documents()` | `AGEGraph.add_graph_documents()` | **동일** |
| `VectorStore.from_texts()` | `Neo4jVector.from_texts()` | `AGEVector.from_texts()` | **동일** |
| `store.similarity_search()` | `Neo4jVector.similarity_search()` | `AGEVector.similarity_search()` | **동일** |
| `store.as_retriever()` | Yes | Yes | **동일** |
| `store.from_existing_graph()` | `Neo4jVector.from_existing_graph()` | `AGEVector.from_existing_graph()` | **동일** |
| `Chain.from_llm()` | `GraphCypherQAChain.from_llm()` | `AGEGraphCypherQAChain.from_llm()` | **동일** |
| `chain.invoke()` | Yes | Yes | **동일** |

## 9. 정리

In [ ]:
# AGE
age_store._drop_table()
age_store.close()
age_graph.close()

# Neo4j — clean up test data
neo4j_graph.query("MATCH (n) DETACH DELETE n")

print("Done.")